# Exact best-response distillation

This experiment asks a narrow question: can the current `(512, 512)` responder network represent a near-perfect best response if it is given exact action labels? A positive result isolates online DQN training—not representational capacity—as the cause of the approximate-BR plateau.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.serialization import load_policy
from liars_poker.policies.neural import NeuralPolicy, compile_neural_to_dense
from liars_poker.policies.neural_q import NeuralQPolicy, compile_neural_q_to_dense
from liars_poker.algo.br_exact_dense_to_dense import best_response_dense
from liars_poker.eval.match_dense import evaluate_dense_response

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cuda':
    print(torch.cuda.get_device_name(0))

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

REFERENCE_RUN_DIR = None
reference_root = REPO_ROOT / 'artifacts' / 'deep_cfr_reference_runs'
if REFERENCE_RUN_DIR is None:
    candidates = sorted(
        (p for p in reference_root.glob(f'{spec.to_short_str()}___*')
         if (p / 'policy' / 'metadata.json').exists()),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError('No completed Deep CFR reference policy found.')
    REFERENCE_RUN_DIR = candidates[0]
else:
    REFERENCE_RUN_DIR = Path(REFERENCE_RUN_DIR)

target, target_spec = load_policy(str(REFERENCE_RUN_DIR / 'policy'))
assert target_spec == spec and isinstance(target, NeuralPolicy)
target.model_p1.to(device)
target.model_p2.to(device)
target.device = torch.device(device)
target.eval()

target_dense = compile_neural_to_dense(target, batch_size=65_536)
exact_br, meta = best_response_dense(
    spec, target_dense, debug=True, store_state_values=False
)
exact_first, exact_second = meta['computer'].exploitability()
exact_exploitability = exact_first + exact_second - 1.0
print('reference:', REFERENCE_RUN_DIR)
print('exact role values:', exact_first, exact_second)
print('exact exploitability:', exact_exploitability)

## Enumerate exact BR labels

Each public history and private hand is one supervised example. Labels are the deterministic actions selected by the exact BR. Training and validation are split independently for each player.

In [ ]:
def make_role_dataset(pid, validation_fraction=0.1, seed=123):
    hids = np.flatnonzero((exact_br.popcount & 1) == pid)
    n_hands = len(exact_br.hands)
    claim_bits = np.arange(exact_br.k, dtype=np.int64)
    history = ((hids[:, None] >> claim_bits[None, :]) & 1).astype(np.float32)
    hand = target.encoder.encode_hands(exact_br.hands, ())[:, :spec.ranks]
    features = np.empty(
        (len(hids), n_hands, target.encoder.input_dim), dtype=np.float32
    )
    features[:, :, :spec.ranks] = hand[None, :, :]
    features[:, :, spec.ranks:] = history[:, None, :]
    features = features.reshape(-1, target.encoder.input_dim)
    labels = exact_br.S[hids].argmax(axis=2).reshape(-1).astype(np.int64)
    legal = np.broadcast_to(
        exact_br.legal_mask[hids, None, :],
        (len(hids), n_hands, exact_br.k + 1),
    ).reshape(-1, exact_br.k + 1).copy()

    rng = np.random.default_rng(seed + pid)
    order = rng.permutation(len(features))
    n_validation = int(round(len(order) * validation_fraction))
    validation = order[:n_validation]
    training = order[n_validation:]
    tensors = {
        'features': torch.from_numpy(features).to(device),
        'labels': torch.from_numpy(labels).to(device),
        'legal': torch.from_numpy(legal).to(device),
        'training': torch.from_numpy(training).to(device),
        'validation': torch.from_numpy(validation).to(device),
    }
    return tensors

datasets = [make_role_dataset(pid) for pid in (0, 1)]
pd.DataFrame({
    f'player {pid + 1}': {
        'training rows': len(data['training']),
        'validation rows': len(data['validation']),
    }
    for pid, data in enumerate(datasets)
}).T

In [ ]:
HIDDEN_SIZES = (512, 512)
BATCH_SIZE = 4096
LEARNING_RATE = 1e-3
TOTAL_STEPS_PER_PLAYER = 10_000
EVALUATE_EVERY = 500
SEED = 7

torch.manual_seed(SEED)
student = NeuralQPolicy(spec, hidden_sizes=HIDDEN_SIZES, device=device)
optimizers = [
    torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    for model in (student.model_p1, student.model_p2)
]
generator = torch.Generator(device=device)
generator.manual_seed(SEED)

def validation_metrics(pid):
    data = datasets[pid]
    idx = data['validation']
    with torch.inference_mode():
        logits = student._model(pid)(data['features'].index_select(0, idx))
        legal = data['legal'].index_select(0, idx)
        labels = data['labels'].index_select(0, idx)
        logits = logits.masked_fill(~legal, -torch.inf)
        loss = F.cross_entropy(logits, labels)
        accuracy = (logits.argmax(dim=1) == labels).float().mean()
    return float(loss), float(accuracy)

def exact_student_value():
    dense = compile_neural_q_to_dense(student, batch_size=65_536)
    p_first, p_second = evaluate_dense_response(spec, target_dense, dense)
    return p_first, p_second, p_first + p_second - 1.0

records = []
start = time.perf_counter()
for step in range(TOTAL_STEPS_PER_PLAYER + 1):
    if step % EVALUATE_EVERY == 0:
        role_metrics = [validation_metrics(pid) for pid in (0, 1)]
        p_first, p_second, discovered = exact_student_value()
        records.append({
            'step per player': step,
            'elapsed_s': time.perf_counter() - start,
            'p_first': p_first,
            'p_second': p_second,
            'discovered exploitability': discovered,
            'exact oracle gap': exact_exploitability - discovered,
            'player 1 validation loss': role_metrics[0][0],
            'player 2 validation loss': role_metrics[1][0],
            'player 1 action accuracy': role_metrics[0][1],
            'player 2 action accuracy': role_metrics[1][1],
        })
        print(
            f'step={step:5d} discovered={discovered:.6f} '
            f'gap={exact_exploitability - discovered:.6f} '
            f'accuracy={[round(x[1], 4) for x in role_metrics]}'
        )
    if step == TOTAL_STEPS_PER_PLAYER:
        break

    for pid in (0, 1):
        data = datasets[pid]
        train_idx = data['training']
        sample = train_idx.index_select(
            0,
            torch.randint(
                len(train_idx), (BATCH_SIZE,), device=device, generator=generator
            ),
        )
        x = data['features'].index_select(0, sample)
        legal = data['legal'].index_select(0, sample)
        labels = data['labels'].index_select(0, sample)
        logits = student._model(pid)(x).masked_fill(~legal, -torch.inf)
        loss = F.cross_entropy(logits, labels)
        optimizers[pid].zero_grad(set_to_none=True)
        loss.backward()
        optimizers[pid].step()

results = pd.DataFrame(records)
results.tail()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(results['step per player'], results['discovered exploitability'], marker='o')
axes[0].axhline(exact_exploitability, color='black', linestyle='--', label='exact BR')
axes[0].set_title('Exact value of distilled responder')
axes[0].legend()
axes[1].semilogy(results['step per player'], results['exact oracle gap'].clip(lower=1e-10), marker='o')
axes[1].set_title('Remaining exact oracle gap')
for pid in (1, 2):
    axes[2].plot(
        results['step per player'], results[f'player {pid} action accuracy'],
        marker='o', label=f'player {pid}',
    )
axes[2].set_title('Held-out action accuracy')
axes[2].legend()
for ax in axes:
    ax.set_xlabel('Optimizer steps per player')
    ax.grid(alpha=0.3)
plt.tight_layout()

pd.Series({
    'exact exploitability': exact_exploitability,
    'final distilled exploitability': results.iloc[-1]['discovered exploitability'],
    'fraction recovered': results.iloc[-1]['discovered exploitability'] / exact_exploitability,
    'final exact oracle gap': results.iloc[-1]['exact oracle gap'],
})